# Stage 01 Only: Speaker Diarization

Notebook này setup đầy đủ môi trường Kaggle, clone đúng nhánh GitHub, chạy đúng một file audio bạn tự nhập đường dẫn, rồi in trực tiếp kết quả speaker diarization. Không chạy music clean, overlap, ASR hay HTML viewer.


## 0. Cấu hình repo, phiên bản và audio

Sửa `AUDIO_INPUT_PATH` cho từng audio bạn muốn test. Các version bên dưới được pin để tránh Kaggle tự kéo lệch torch/NVIDIA stack hoặc NeMo.


In [ ]:
from pathlib import Path

# =========================
# 0. GitHub branch controls
# =========================
REPO_URL = "https://github.com/tuanad121/sommelier.git"
BRANCH = "kaggle-gpu"
FORCE_RECLONE = True

# =========================
# 1. Manual single-audio input
# =========================
# Sửa dòng này cho từng audio bạn muốn test.
# Ví dụ: AUDIO_INPUT_PATH = '/kaggle/input/my-audio-folder/sample.wav'
AUDIO_INPUT_PATH = '/kaggle/input/YOUR_DATASET/YOUR_AUDIO.wav'
AUDIO_INPUT_EXTENSIONS = ('.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.opus')

# =========================
# 2. Kaggle paths
# =========================
PROJECT_ROOT = Path('/kaggle/working/sommelier')
PIPELINE_DIR = PROJECT_ROOT / 'podcast-pipeline'
CONFIG_PATH = PIPELINE_DIR / 'config.json'
BATCH_ROOT = Path('/kaggle/working/sommelier_batch_outputs')
OUTPUT_ROOT = BATCH_ROOT / 'diarization_only_runs'
SETUP_LOG_DIR = BATCH_ROOT / 'setup_logs'

# =========================
# 3. Runtime / GPU controls
# =========================
CUDA_VISIBLE_DEVICES = '0'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'
REQUIRE_GPU = True
PRINT_NVIDIA_SMI = True
DIAR_DEVICE_INDEX = 0
SORTFORMER_DEVICE_INDEX = 0
SPEAKER_LINK_THRESHOLD = 0.75
MIN_SPLIT_SILENCE = 1.0
MAX_DIA_CHUNK_DURATION = 900.0
AUDIO_GAIN_CLAMP_DB = 6.0  # Doi 3.0/6.0/9.0 de test gain normalization

# Speaker resegmentation audit controls.
# Chạy local pyannote quanh vùng nghi ngờ, map về speaker global bằng embedding, rồi chỉ sửa khi confidence cao.
SPEAKER_RESEGMENTATION_AUDIT = True
SPEAKER_RESEGMENTATION_METHOD = "sliding_window"
SLIDING_WINDOW_SIZE = 0.2
SLIDING_STEP_SIZE = 0.1
SLIDING_THRESHOLD_HIGH = 0.75
SLIDING_THRESHOLD_LOW = 0.55
RESEGMENTATION_PYANNOTE_MODEL_NAME = "pyannote/speaker-diarization-3.1"
RESEGMENTATION_BOUNDARY_WINDOW = 1.5
RESEGMENTATION_INTERIOR_MIN_DURATION = 8.0
RESEGMENTATION_MAX_SHIFT = 0.8
RESEGMENTATION_MAX_EXTEND = 0.8
RESEGMENTATION_MIN_DURATION = 0.3
RESEGMENTATION_MIN_OVERLAP_DURATION = 0.2
RESEGMENTATION_MIN_MAPPING_SCORE = 0.7
RESEGMENTATION_MIN_MAPPING_MARGIN = 0.08
RESEGMENTATION_REFERENCE_MIN_SEGMENT = 2.0
RESEGMENTATION_REFERENCE_MAX_SEGMENTS = 6

# Sortformer inference/post-processing controls.
# Mặc định thử Streaming Sortformer v2.1 để dùng speaker cache nội bộ.
# Nếu muốn quay lại offline v1: SORTFORMER_MODEL_NAME = "nvidia/diar_sortformer_4spk-v1" và SORTFORMER_STREAMING_CONFIG = False.
SORTFORMER_MODEL_NAME = "nvidia/diar_streaming_sortformer_4spk-v2.1"
# CallHome-part1 chính thức: onset=0.53, offset=0.49, pad_onset=0.23, pad_offset=0.01, min_duration_on=0.42, min_duration_off=0.34.
SORTFORMER_BATCH_SIZE = 1
SORTFORMER_NUM_WORKERS = 0
SORTFORMER_STREAMING_CONFIG = True
SORTFORMER_CHUNK_LEN = 340
SORTFORMER_CHUNK_LEFT_CONTEXT = 1
SORTFORMER_CHUNK_RIGHT_CONTEXT = 40
SORTFORMER_FIFO_LEN = 40
SORTFORMER_SPKCACHE_UPDATE_PERIOD = 300
SORTFORMER_SPKCACHE_LEN = 200
SORTFORMER_POSTPROCESSING = True
SORTFORMER_POSTPROCESSING_YAML = OUTPUT_ROOT / 'sortformer_postprocessing.yaml'
SORTFORMER_PP_ONSET = 0.3
SORTFORMER_PP_OFFSET = 0.33
SORTFORMER_PP_PAD_ONSET = 0.015
SORTFORMER_PP_PAD_OFFSET = 0.015
SORTFORMER_PP_MIN_DURATION_ON = 0.35
SORTFORMER_PP_MIN_DURATION_OFF = 0.35

# =========================
# 4. Pinned package versions
# =========================
INSTALL_DEPENDENCIES = True
TORCH_PACKAGE = "torch==2.7.1"
TORCHAUDIO_PACKAGE = "torchaudio==2.7.1"
TORCHVISION_PACKAGE = "torchvision==0.22.1"
PYTORCH_WHEEL_EXTRA_INDEX_URL = "https://download.pytorch.org/whl/cu126"
LIGHTNING_PACKAGE = "lightning==2.4.0"
PYTORCH_LIGHTNING_PACKAGE = "pytorch-lightning==2.5.2"
NEMO_PACKAGE = "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@main"
NUMPY_PACKAGE = "numpy==2.2.6"
NUMBA_PACKAGE = "numba==0.61.2"
LLVMLITE_PACKAGE = "llvmlite==0.44.0"
TORCHMETRICS_PACKAGE = "torchmetrics==1.7.4"
PILLOW_PACKAGE = "pillow<12.0"
CTRANSLATE2_PACKAGE = "ctranslate2==4.6.3"

# Extra model names downloaded during stage 01.
SPEAKER_EMBEDDING_MODEL_NAME = "pyannote/embedding"
SILERO_VAD_REPO = "snakers4/silero-vad"

# =========================
# 5. Secrets
# =========================
HF_SECRET_NAME = "HF_TOKEN"

AUDIO_PATH = Path(AUDIO_INPUT_PATH)
if not AUDIO_INPUT_PATH or 'YOUR_AUDIO' in AUDIO_INPUT_PATH or not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Hãy sửa AUDIO_INPUT_PATH thành đường dẫn audio thật trên Kaggle: {AUDIO_INPUT_PATH}')
if AUDIO_PATH.suffix.lower() not in AUDIO_INPUT_EXTENSIONS:
    raise ValueError(f'Định dạng audio chưa hỗ trợ: {AUDIO_PATH.suffix}. Hỗ trợ: {AUDIO_INPUT_EXTENSIONS}')

RUN_DIR = OUTPUT_ROOT / f'run_full_{AUDIO_PATH.stem}'

print('REPO_URL =', REPO_URL)
print('BRANCH =', BRANCH)
print('AUDIO_INPUT_PATH =', AUDIO_PATH)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RUN_DIR =', RUN_DIR)
print('Pinned packages:')
for name in [
    TORCH_PACKAGE, TORCHAUDIO_PACKAGE, TORCHVISION_PACKAGE, LIGHTNING_PACKAGE,
    PYTORCH_LIGHTNING_PACKAGE, NEMO_PACKAGE, NUMPY_PACKAGE, NUMBA_PACKAGE,
    LLVMLITE_PACKAGE, TORCHMETRICS_PACKAGE, CTRANSLATE2_PACKAGE,
]:
    print('  -', name)
print('Models:')
print('  -', SORTFORMER_MODEL_NAME)
print('  -', SPEAKER_EMBEDDING_MODEL_NAME)
print('  -', SILERO_VAD_REPO)
print('Sortformer post-processing:')
print('  enabled =', SORTFORMER_POSTPROCESSING)
print('  yaml =', SORTFORMER_POSTPROCESSING_YAML)
print('  batch_size =', SORTFORMER_BATCH_SIZE)
print('  num_workers =', SORTFORMER_NUM_WORKERS)
print('  streaming_config =', SORTFORMER_STREAMING_CONFIG)
print('  chunk_len =', SORTFORMER_CHUNK_LEN)
print('  chunk_left_context =', SORTFORMER_CHUNK_LEFT_CONTEXT)
print('  chunk_right_context =', SORTFORMER_CHUNK_RIGHT_CONTEXT)
print('  fifo_len =', SORTFORMER_FIFO_LEN)
print('  spkcache_update_period =', SORTFORMER_SPKCACHE_UPDATE_PERIOD)
print('  spkcache_len =', SORTFORMER_SPKCACHE_LEN)
print('  min_split_silence =', MIN_SPLIT_SILENCE)
print('  max_dia_chunk_duration =', MAX_DIA_CHUNK_DURATION)
print('  audio_gain_clamp_db = +/-', AUDIO_GAIN_CLAMP_DB)
print('  speaker_resegmentation_audit =', SPEAKER_RESEGMENTATION_AUDIT)
print('  speaker_resegmentation_method =', SPEAKER_RESEGMENTATION_METHOD)
print('  sliding_window_size =', SLIDING_WINDOW_SIZE)
print('  sliding_step_size =', SLIDING_STEP_SIZE)
print('  sliding_threshold_high =', SLIDING_THRESHOLD_HIGH)
print('  sliding_threshold_low =', SLIDING_THRESHOLD_LOW)
print('  resegmentation_pyannote_model_name =', RESEGMENTATION_PYANNOTE_MODEL_NAME)
print('  resegmentation_boundary_window =', RESEGMENTATION_BOUNDARY_WINDOW)
print('  resegmentation_interior_min_duration =', RESEGMENTATION_INTERIOR_MIN_DURATION)
print('  resegmentation_max_shift =', RESEGMENTATION_MAX_SHIFT)
print('  resegmentation_max_extend =', RESEGMENTATION_MAX_EXTEND)
print('  resegmentation_min_duration =', RESEGMENTATION_MIN_DURATION)
print('  resegmentation_min_overlap_duration =', RESEGMENTATION_MIN_OVERLAP_DURATION)
print('  resegmentation_min_mapping_score =', RESEGMENTATION_MIN_MAPPING_SCORE)
print('  resegmentation_min_mapping_margin =', RESEGMENTATION_MIN_MAPPING_MARGIN)
print('  resegmentation_reference_min_segment =', RESEGMENTATION_REFERENCE_MIN_SEGMENT)
print('  resegmentation_reference_max_segments =', RESEGMENTATION_REFERENCE_MAX_SEGMENTS)
print('  onset =', SORTFORMER_PP_ONSET)
print('  offset =', SORTFORMER_PP_OFFSET)
print('  pad_onset =', SORTFORMER_PP_PAD_ONSET)
print('  pad_offset =', SORTFORMER_PP_PAD_OFFSET)
print('  min_duration_on =', SORTFORMER_PP_MIN_DURATION_ON)
print('  min_duration_off =', SORTFORMER_PP_MIN_DURATION_OFF)


## 1. Helper chạy lệnh và thiết lập GPU


In [ ]:
import os
import shlex
import subprocess
import time
from pathlib import Path

for _dir in [BATCH_ROOT, OUTPUT_ROOT, SETUP_LOG_DIR]:
    Path(_dir).mkdir(parents=True, exist_ok=True)

if CUDA_VISIBLE_DEVICES is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_VISIBLE_DEVICES)
if PYTORCH_CUDA_ALLOC_CONF:
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = str(PYTORCH_CUDA_ALLOC_CONF)

print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<not set>'))
print('PYTORCH_CUDA_ALLOC_CONF =', os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '<not set>'))
print('DIAR_DEVICE_INDEX =', DIAR_DEVICE_INDEX)
print('SORTFORMER_DEVICE_INDEX =', SORTFORMER_DEVICE_INDEX)


def _format_cmd(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd) if isinstance(cmd, (list, tuple)) else str(cmd)


def tail_file(path, n=30):
    path = Path(path)
    if not path.exists():
        return ''
    lines = path.read_text(encoding='utf-8', errors='replace').splitlines()
    return '\n'.join(lines[-n:])


def run_logged(cmd, log_name, cwd='/kaggle/working', env=None, shell=False, tail=20):
    log_path = SETUP_LOG_DIR / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Running:', _format_cmd(cmd))
    print('Log:', log_path)
    start = time.time()
    with open(log_path, 'w', encoding='utf-8', errors='replace') as log:
        proc = subprocess.run(
            cmd,
            cwd=str(cwd),
            env=env,
            shell=shell,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    print('Exit code:', proc.returncode, '| seconds:', round(time.time() - start, 2))
    log_tail = tail_file(log_path, n=tail)
    if log_tail:
        print(f'--- last {tail} log lines ---')
        print(log_tail)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return log_path


## 2. Clone đúng nhánh GitHub


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

os.chdir('/kaggle/working')
if PROJECT_ROOT.exists() and FORCE_RECLONE:
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    run_logged(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], '01_clone_repo.log', tail=40)
else:
    print('Repo already exists:', PROJECT_ROOT)
    run_logged(['git', 'fetch', 'origin', BRANCH], '01_fetch_repo.log', cwd=PROJECT_ROOT, tail=30)
    run_logged(['git', 'checkout', BRANCH], '01_checkout_branch.log', cwd=PROJECT_ROOT, tail=30)

os.chdir(PIPELINE_DIR)
print('cwd:', os.getcwd())
print('branch:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], text=True).strip())
print('commit:', subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip())


## 3. Cài dependencies đã pin version

Cell này cần Internet. Nếu bạn chạy lại trong cùng Kaggle session và môi trường đã ổn, có thể đặt `INSTALL_DEPENDENCIES = False` ở cell cấu hình.


In [ ]:
import importlib
import os
from pathlib import Path

os.chdir(PIPELINE_DIR)

if INSTALL_DEPENDENCIES:
    run_logged(['apt-get', 'update', '-y'], '02_apt_update.log', tail=10)
    run_logged(['apt-get', 'install', '-y', 'ffmpeg', 'git', 'git-lfs', 'libaio-dev'], '03_apt_install.log', tail=10)
    run_logged(['python', '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel', 'packaging', 'ninja'], '04_pip_base.log', tail=12)

    req = Path('requirements.txt').read_text(encoding='utf-8')
    filtered = []
    skip_prefixes = ('torch==', 'torchaudio==', 'torchvision==', 'ctranslate2==')
    for line in req.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        if 'nemo-toolkit[all]' in stripped:
            continue
        if stripped.startswith(skip_prefixes):
            continue
        filtered.append(line)
    REQUIREMENTS_PATH = PIPELINE_DIR / 'requirements-kaggle-diarization.txt'
    REQUIREMENTS_PATH.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
    run_logged(['python', '-m', 'pip', 'install', '-r', str(REQUIREMENTS_PATH)], '05_pip_requirements_diarization.log', tail=24)

    run_logged(['python', '-m', 'pip', 'uninstall', '-y', 'nemo-toolkit', 'lightning', 'pytorch-lightning'], '06_pip_uninstall_nemo.log', tail=10)
    run_logged(['python', '-m', 'pip', 'install', LIGHTNING_PACKAGE, PYTORCH_LIGHTNING_PACKAGE], '07_pip_lightning.log', tail=16)
    run_logged(['python', '-m', 'pip', 'install', NEMO_PACKAGE], '08_pip_nemo_asr.log', tail=24)

    torch_stack_cmd = ['python', '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall']
    if PYTORCH_WHEEL_EXTRA_INDEX_URL:
        torch_stack_cmd.extend(['--extra-index-url', PYTORCH_WHEEL_EXTRA_INDEX_URL])
    torch_stack_cmd.extend([TORCH_PACKAGE, TORCHAUDIO_PACKAGE, TORCHVISION_PACKAGE])
    run_logged(torch_stack_cmd, '09_pip_torch_stack.log', tail=30)

    run_logged(['python', '-m', 'pip', 'install', PILLOW_PACKAGE], '10_pip_pillow.log', tail=10)
    run_logged(['python', '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', '--no-deps', TORCHMETRICS_PACKAGE], '11_pip_torchmetrics.log', tail=10)
    run_logged(['python', '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', NUMPY_PACKAGE, NUMBA_PACKAGE, LLVMLITE_PACKAGE], '12_pip_numpy_numba.log', tail=16)
    run_logged(['python', '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', '--no-deps', CTRANSLATE2_PACKAGE], '13_pip_ctranslate2.log', tail=16)
else:
    print('INSTALL_DEPENDENCIES=False, bỏ qua cài dependencies.')


def configure_cuda_library_paths():
    module_names = [
        'nvidia.cublas.lib',
        'nvidia.cuda_runtime.lib',
        'nvidia.cuda_nvrtc.lib',
        'nvidia.cudnn.lib',
        'nvidia.cufft.lib',
        'nvidia.curand.lib',
        'nvidia.cusolver.lib',
        'nvidia.cusparse.lib',
        'nvidia.nccl.lib',
    ]
    paths = []
    for module_name in module_names:
        try:
            module = importlib.import_module(module_name)
            paths.append(Path(module.__file__).parent)
        except Exception as exc:
            print(f'Skip CUDA lib path {module_name}: {exc}')
    paths.extend([Path('/usr/local/nvidia/lib64'), Path('/usr/local/cuda/lib64')])
    seen = set()
    result = []
    for path in paths:
        path = str(path)
        if path and path not in seen and Path(path).exists():
            seen.add(path)
            result.append(path)
    os.environ['LD_LIBRARY_PATH'] = ':'.join(result + [os.environ.get('LD_LIBRARY_PATH', '')]).rstrip(':')
    os.environ.pop('LD_PRELOAD', None)
    print('LD_LIBRARY_PATH =', os.environ['LD_LIBRARY_PATH'])
    print('LD_PRELOAD cleared for CTranslate2 4.6.3')
    return result

configure_cuda_library_paths()


## 4. Kiểm tra phiên bản runtime


In [ ]:
import importlib.metadata as importlib_metadata
import subprocess
import numpy
import numba
import torch
import torchvision
import torchaudio

packages_to_report = [
    'nemo-toolkit',
    'pyannote.audio',
    'lightning',
    'pytorch-lightning',
    'torchmetrics',
    'numpy',
    'numba',
    'librosa',
    'onnxruntime',
    'pandas',
    'torch',
    'torchvision',
    'torchaudio',
]
for pkg in packages_to_report:
    try:
        if pkg == 'numpy':
            version = numpy.__version__
        elif pkg == 'numba':
            version = numba.__version__
        elif pkg == 'torch':
            version = torch.__version__
        elif pkg == 'torchvision':
            version = torchvision.__version__
        elif pkg == 'torchaudio':
            version = torchaudio.__version__
        else:
            version = importlib_metadata.version(pkg)
        print(pkg + ':', version)
    except Exception as exc:
        print(pkg + ':', 'missing', exc)

print('Torch stack:', torch.__version__, torchvision.__version__, torchaudio.__version__)
print('CUDA:', torch.cuda.is_available())
visible_gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('Visible CUDA device count:', visible_gpu_count)
if torch.cuda.is_available():
    for i in range(visible_gpu_count):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
    invalid_assignments = {
        name: idx for name, idx in {
            'diar/vad/speaker-link': DIAR_DEVICE_INDEX,
            'sortformer': SORTFORMER_DEVICE_INDEX,
        }.items()
        if idx is not None and idx >= visible_gpu_count
    }
    if invalid_assignments:
        raise ValueError(f'GPU index không hợp lệ. Kaggle chỉ thấy {visible_gpu_count} GPU: {invalid_assignments}')
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('REQUIRE_GPU=True nhưng torch.cuda.is_available() = False. Hãy bật Kaggle GPU hoặc đặt REQUIRE_GPU=False.')
if PRINT_NVIDIA_SMI:
    subprocess.run(['nvidia-smi'], check=False)


## 5. Gắn Hugging Face token vào config

Tạo Kaggle Secret tên `HF_TOKEN`. Token này dùng cho `pyannote/embedding`; Sortformer và Silero VAD sẽ được tải ở bước chạy diarization.


In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

try:
    token = UserSecretsClient().get_secret(HF_SECRET_NAME)
except Exception as exc:
    raise RuntimeError(f'Không đọc được Kaggle Secret {HF_SECRET_NAME}. Hãy tạo secret HF_TOKEN trước khi chạy.') from exc

if not token:
    raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME} đang trống.')

login(token=token, add_to_git_credential=False)
print('HF token:', token[:8] + '...')
try:
    print(whoami(token=token))
except Exception as exc:
    print('HF whoami failed, nhưng token vẫn được ghi vào config:', exc)

cfg = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
cfg['huggingface_token'] = token
cfg.setdefault('entrypoint', {})['SAMPLE_RATE'] = 16000
cfg['entrypoint']['input_folder_path'] = str(AUDIO_PATH.parent)
CONFIG_PATH.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding='utf-8')
print('config.json updated:', CONFIG_PATH)


## 6. Chạy speaker diarization cho một audio


In [ ]:
sortformer_pp = {
    'onset': SORTFORMER_PP_ONSET,
    'offset': SORTFORMER_PP_OFFSET,
    'pad_onset': SORTFORMER_PP_PAD_ONSET,
    'pad_offset': SORTFORMER_PP_PAD_OFFSET,
    'min_duration_on': SORTFORMER_PP_MIN_DURATION_ON,
    'min_duration_off': SORTFORMER_PP_MIN_DURATION_OFF,
}
sortformer_pp_payload = {'parameters': sortformer_pp}

if SORTFORMER_POSTPROCESSING:
    SORTFORMER_POSTPROCESSING_YAML.parent.mkdir(parents=True, exist_ok=True)
    yaml_lines = ['parameters:']
    yaml_lines.extend(f'  {key}: {value}' for key, value in sortformer_pp.items())
    SORTFORMER_POSTPROCESSING_YAML.write_text('\n'.join(yaml_lines) + '\n', encoding='utf-8')
    print('Sortformer postprocessing YAML =', SORTFORMER_POSTPROCESSING_YAML)
    print(sortformer_pp_payload)
else:
    print('SORTFORMER_POSTPROCESSING=False, dùng post-processing mặc định của NeMo.')

cmd = [
    'python', str(PIPELINE_DIR / 'run_stage_diarization_only.py'),
    '--input_audio_path', str(AUDIO_PATH),
    '--config_path', str(CONFIG_PATH),
    '--output_root', str(OUTPUT_ROOT),
    '--vad',
    '--speaker-link-threshold', str(SPEAKER_LINK_THRESHOLD),
    '--min_split_silence', str(MIN_SPLIT_SILENCE),
    '--max_dia_chunk_duration', str(MAX_DIA_CHUNK_DURATION),
    '--audio-gain-clamp-db', str(AUDIO_GAIN_CLAMP_DB),
    '--diar_device_index', str(DIAR_DEVICE_INDEX),
    '--sortformer_device_index', str(SORTFORMER_DEVICE_INDEX),
    '--sortformer_model_name', str(SORTFORMER_MODEL_NAME),
    '--sortformer_batch_size', str(SORTFORMER_BATCH_SIZE),
    '--sortformer_num_workers', str(SORTFORMER_NUM_WORKERS),
]
if SPEAKER_RESEGMENTATION_AUDIT:
    cmd.extend([
        '--speaker-resegmentation-audit',
        '--speaker-resegmentation-method', str(SPEAKER_RESEGMENTATION_METHOD),
        '--sliding-window-size', str(SLIDING_WINDOW_SIZE),
        '--sliding-step-size', str(SLIDING_STEP_SIZE),
        '--sliding-threshold-high', str(SLIDING_THRESHOLD_HIGH),
        '--sliding-threshold-low', str(SLIDING_THRESHOLD_LOW),
        '--resegmentation-pyannote-model-name', str(RESEGMENTATION_PYANNOTE_MODEL_NAME),
        '--resegmentation-boundary-window', str(RESEGMENTATION_BOUNDARY_WINDOW),
        '--resegmentation-interior-min-duration', str(RESEGMENTATION_INTERIOR_MIN_DURATION),
        '--resegmentation-max-shift', str(RESEGMENTATION_MAX_SHIFT),
        '--resegmentation-max-extend', str(RESEGMENTATION_MAX_EXTEND),
        '--resegmentation-min-duration', str(RESEGMENTATION_MIN_DURATION),
        '--resegmentation-min-overlap-duration', str(RESEGMENTATION_MIN_OVERLAP_DURATION),
        '--resegmentation-min-mapping-score', str(RESEGMENTATION_MIN_MAPPING_SCORE),
        '--resegmentation-min-mapping-margin', str(RESEGMENTATION_MIN_MAPPING_MARGIN),
        '--resegmentation-reference-min-segment', str(RESEGMENTATION_REFERENCE_MIN_SEGMENT),
        '--resegmentation-reference-max-segments', str(RESEGMENTATION_REFERENCE_MAX_SEGMENTS),
    ])
if SORTFORMER_STREAMING_CONFIG:
    cmd.extend([
        '--sortformer-streaming-config',
        '--sortformer_chunk_len', str(SORTFORMER_CHUNK_LEN),
        '--sortformer_chunk_left_context', str(SORTFORMER_CHUNK_LEFT_CONTEXT),
        '--sortformer_chunk_right_context', str(SORTFORMER_CHUNK_RIGHT_CONTEXT),
        '--sortformer_fifo_len', str(SORTFORMER_FIFO_LEN),
        '--sortformer_spkcache_update_period', str(SORTFORMER_SPKCACHE_UPDATE_PERIOD),
        '--sortformer_spkcache_len', str(SORTFORMER_SPKCACHE_LEN),
    ])
if SORTFORMER_POSTPROCESSING:
    cmd.extend([
        '--sortformer-postprocessing',
        '--sortformer-postprocessing-yaml', str(SORTFORMER_POSTPROCESSING_YAML),
        '--sortformer-pp-onset', str(SORTFORMER_PP_ONSET),
        '--sortformer-pp-offset', str(SORTFORMER_PP_OFFSET),
        '--sortformer-pp-pad-onset', str(SORTFORMER_PP_PAD_ONSET),
        '--sortformer-pp-pad-offset', str(SORTFORMER_PP_PAD_OFFSET),
        '--sortformer-pp-min-duration-on', str(SORTFORMER_PP_MIN_DURATION_ON),
        '--sortformer-pp-min-duration-off', str(SORTFORMER_PP_MIN_DURATION_OFF),
    ])
run_logged(cmd, f'20_stage_diarization_{AUDIO_PATH.stem}.log', cwd=PIPELINE_DIR, tail=50)


## 7. In kết quả trực tiếp trên notebook


In [ ]:
import json
import pandas as pd

DIARIZATION_JSON = RUN_DIR / '01_diarization' / 'diarization.json'
if not DIARIZATION_JSON.exists():
    raise FileNotFoundError(f'Không tìm thấy diarization output: {DIARIZATION_JSON}')

payload = json.loads(DIARIZATION_JSON.read_text(encoding='utf-8'))
segments = payload.get('segments', [])
metadata = payload.get('metadata', {})

print('diarization_json =', DIARIZATION_JSON)
print('audio_duration_seconds =', metadata.get('audio_duration_seconds'))
print('processing_time_seconds =', metadata.get('processing_time_seconds'))
print('rt_factor =', metadata.get('rt_factor'))
print('segment_count =', len(segments))

df = pd.DataFrame(segments)
if df.empty:
    print('Không có segment diarization nào.')
else:
    df['duration'] = (df['end'].astype(float) - df['start'].astype(float)).round(3)
    display(df[['index', 'start', 'end', 'duration', 'speaker']])

    speaker_summary = (
        df.groupby('speaker', dropna=False)
        .agg(segments=('speaker', 'size'), total_seconds=('duration', 'sum'))
        .reset_index()
        .sort_values('speaker')
    )
    speaker_summary['total_seconds'] = speaker_summary['total_seconds'].round(3)
    display(speaker_summary)


## 8. Nghe full audio và từng segment ngay trong notebook

Cell này tạo audio player cho full audio, sau đó cắt từng segment diarization thành file WAV nhỏ để nghe và đánh giá trực tiếp.

In [ ]:
from IPython.display import Audio, HTML, display
from pydub import AudioSegment

FULL_AUDIO_PATH = RUN_DIR / '00_input' / 'full.wav'
SEGMENT_AUDIO_DIR = RUN_DIR / '01_diarization' / 'segment_audio_preview'
SEGMENT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

if not FULL_AUDIO_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy full audio để nghe: {FULL_AUDIO_PATH}')

display(HTML('<h3>Full audio</h3>'))
full_audio = Audio(str(FULL_AUDIO_PATH))
display(full_audio)

if df.empty:
    print('Không có segment để tạo audio player.')
else:
    full_audio_segment = AudioSegment.from_file(FULL_AUDIO_PATH)
    segment_audio_rows = []

    for _, row in df.iterrows():
        index = str(row['index'])
        speaker = str(row['speaker'])
        start = float(row['start'])
        end = float(row['end'])
        duration = float(row['duration'])
        start_ms = max(0, int(round(start * 1000)))
        end_ms = max(start_ms, int(round(end * 1000)))
        segment_path = SEGMENT_AUDIO_DIR / f'{index}_{speaker}.wav'
        full_audio_segment[start_ms:end_ms].export(segment_path, format='wav')
        segment_audio_rows.append({
            'index': index,
            'start': start,
            'end': end,
            'duration': duration,
            'speaker': speaker,
            'segment_audio': str(segment_path),
        })

    segment_audio_df = pd.DataFrame(segment_audio_rows)
    display(HTML('<h3>Segment audio players</h3>'))
    display(segment_audio_df[['index', 'start', 'end', 'duration', 'speaker', 'segment_audio']])

    for item in segment_audio_rows:
        label = (
            f"<b>{item['index']}</b> | {item['speaker']} | "
            f"{item['start']:.3f}s - {item['end']:.3f}s | "
            f"duration {item['duration']:.3f}s"
        )
        display(HTML(label))
        display(Audio(item['segment_audio']))
